# StackOverflow Forward Indexing

This notebook creates forward indices for StackOverflow collections already generated in `demo/collections`.

Run this after your conversion/indexing notebook has created collections like `stackoverflow_train`, `stackoverflow_test`, etc.

In [3]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()

# Set to one collection name, e.g. ['stackoverflow_train'], or keep all
TARGET_COLLECTIONS = [
    'stackoverflow_train',
    'stackoverflow_test',
    'stackoverflow_dev1',
    'stackoverflow_dev2',
    'stackoverflow_tran'
]

# Field definitions for create_fwd_index.sh
FIELD_DEFS = [
    'text:parsedText',
    'text_unlemm:parsedText',
    'text_raw:textRaw',
    # 'bigram:parsedBOW' # ?
]

# If True, rebuild index files for each field from scratch
CLEAN_EXISTING = False

print('REPO_ROOT =', REPO_ROOT)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('COLLECT_ROOT =', COLLECT_ROOT)
print('TARGET_COLLECTIONS =', TARGET_COLLECTIONS)
print('FIELD_DEFS =', FIELD_DEFS)
print('CLEAN_EXISTING =', CLEAN_EXISTING)

REPO_ROOT = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
SCRIPTS_DIR = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
COLLECT_ROOT = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
TARGET_COLLECTIONS = ['stackoverflow_train', 'stackoverflow_test', 'stackoverflow_dev1', 'stackoverflow_dev2', 'stackoverflow_tran']
FIELD_DEFS = ['text:parsedText', 'text_unlemm:parsedText', 'text_raw:textRaw']
CLEAN_EXISTING = False


## 1) Validate collection layout

Each target collection should have one split subfolder under `input_data` containing `QuestionFields.jsonl`, `AnswerFields.jsonl`, and `qrels.txt`.

In [4]:
for coll in TARGET_COLLECTIONS:
    coll_dir = COLLECT_ROOT / coll
    if not coll_dir.exists():
        raise FileNotFoundError(f'Missing collection directory: {coll_dir}')

    input_data_dir = coll_dir / 'input_data'
    if not input_data_dir.exists():
        raise FileNotFoundError(f'Missing input_data directory: {input_data_dir}')

    split_subdirs = sorted([p for p in input_data_dir.iterdir() if p.is_dir()])
    if not split_subdirs:
        raise FileNotFoundError(f'No split subdirectory found in {input_data_dir}')

    print(f'{coll}: split dirs -> {[p.name for p in split_subdirs]}')

    for split_dir in split_subdirs:
        needed = [
            split_dir / 'QuestionFields.jsonl',
            split_dir / 'AnswerFields.jsonl',
            split_dir / 'qrels.txt'
        ]
        for p in needed:
            if not p.exists():
                raise FileNotFoundError(f'Missing required file: {p}')

print('All target collections have the expected structure.')

stackoverflow_train: split dirs -> ['train']
stackoverflow_test: split dirs -> ['test']
stackoverflow_dev1: split dirs -> ['dev1']
stackoverflow_dev2: split dirs -> ['dev2']
stackoverflow_tran: split dirs -> ['tran']
All target collections have the expected structure.


## 2) Create forward indices

This runs `create_fwd_index.sh` for each selected collection and each field definition.

In [5]:
env = os.environ.copy()
env['COLLECT_ROOT'] = str(COLLECT_ROOT)

for coll in TARGET_COLLECTIONS:
    print(f'\n===== Building forward indices for {coll} =====')
    for field_def in FIELD_DEFS:
        cmd = ['./index/create_fwd_index.sh', coll, field_def]
        if CLEAN_EXISTING:
            cmd.append('-clean')

        print('Running:', ' '.join(cmd))
        res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=env, text=True, capture_output=True)
        print(res.stdout)
        if res.returncode != 0:
            print(res.stderr)
            raise RuntimeError(f'create_fwd_index.sh failed for {coll} {field_def} with code {res.returncode}')


===== Building forward indices for stackoverflow_train =====
Running: ./index/create_fwd_index.sh stackoverflow_train text:parsedText
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
Input data directory:      /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_train/input_data
Forward index directory:   /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_train/forward_index
Remove old index?:         0
Field definition:          text:parsedText
Index type arguments:        
Checking input sub-directory: train
Found indexable data file: train/AnswerFields.jsonl
Found query file: train/QuestionFields.jsonl
JAVA_OPTS=-Xms4070758k -Xmx28495306k -server

Running: ./index/create_fwd_index.sh stackoverflow_train text_unlemm:parsedText
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNe

## 3) Verify output

Checks that each target collection now has a `forward_index` directory.

In [6]:
for coll in TARGET_COLLECTIONS:
    fwd_dir = COLLECT_ROOT / coll / 'forward_index'
    print(f'\n{coll}:')
    print('  forward_index exists:', fwd_dir.exists())
    if fwd_dir.exists():
        children = sorted([p.name for p in fwd_dir.iterdir()])
        print('  files sample:', children[:20])


stackoverflow_train:
  forward_index exists: True
  files sample: ['text', 'text.mapdb_dataDict', 'text_raw', 'text_raw.mapdb_dataDict', 'text_unlemm', 'text_unlemm.mapdb_dataDict']

stackoverflow_test:
  forward_index exists: True
  files sample: ['text', 'text.mapdb_dataDict', 'text_raw', 'text_raw.mapdb_dataDict', 'text_unlemm', 'text_unlemm.mapdb_dataDict']

stackoverflow_dev1:
  forward_index exists: True
  files sample: ['text', 'text.mapdb_dataDict', 'text_raw', 'text_raw.mapdb_dataDict', 'text_unlemm', 'text_unlemm.mapdb_dataDict']

stackoverflow_dev2:
  forward_index exists: True
  files sample: ['text', 'text.mapdb_dataDict', 'text_raw', 'text_raw.mapdb_dataDict', 'text_unlemm', 'text_unlemm.mapdb_dataDict']

stackoverflow_tran:
  forward_index exists: True
  files sample: ['text', 'text.mapdb_dataDict', 'text_raw', 'text_raw.mapdb_dataDict', 'text_unlemm', 'text_unlemm.mapdb_dataDict']


## Notes

- To run only one split collection, set `TARGET_COLLECTIONS = ['stackoverflow_train']` (or another one).
- To rebuild from scratch, set `CLEAN_EXISTING = True`.
- Field defs used here match the StackOverflow JSONL schema produced by your conversion notebook.